# Merger tree and host mass accretion history

One figure, two panels, one low-resolution SatGen tree of a $10^{12}\,M_\odot$ host (read from `trees/`, nothing is regrown):

1. **Merger tree.** A node-link tree in the style of `Tree_Vis.plot_acc_tree`: nodes sized by mass, coloured by order, joined by edges. Here it is laid on its side so that time runs along x. The host's main branch is the trunk. Every other branch is a limb that starts at a node (placed where the branch formed) and runs diagonally into the branch it merges with, at the time it merges. Sub-branches come off their parent limb the same way.
2. **Host mass accretion history**, $M_{\rm host}(t)$, on the same time axis, $M_{\rm host}$ in $M_\odot$ (log axis) against $\log(1+z)$.

The figure is drawn twice: once with only the first-order branches, and once with the higher-order branches added.

Only first-order ($k=1$) limbs reach the trunk, so they are the ones that make $M_{\rm host}$ jump. Each one meets the trunk at a host node; a dotted guide drops from that node, and the jump at the same time in the lower panel is drawn as a thick segment. The largest mergers carry the same number in both panels.

Higher-order branches ($k\geq2$) merge with another branch before it reaches the host. Their mass arrives inside their $k=1$ parent, so they have no jump of their own.

The curve also rises between jumps. That is smooth accretion: mass in progenitors below the tree resolution `psi_res`.

Styled to match `schematic_figures.ipynb` (DejaVu Sans + mathtext, 16 pt labels, 12 pt inward ticks); the colours are this notebook's own.

This notebook needs only `numpy` and `matplotlib`. It reads the `.npz` directly, with the same conventions as `Tree_Reader` (drop the dead row 1, shift `ParentID`, accretion index = `nanargmax` of the mass).

In [1]:
import os
import warnings; warnings.simplefilter('ignore')

from types import SimpleNamespace

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

SATGEN = os.environ.get('SATGEN', '/Users/jsmonzon/Research/SatGen')

# ---- shared style: same as schematic_figures.ipynb (default DejaVu Sans + mathtext,
#      16 pt labels, 12 pt inward ticks, no top/right spines). Colours are this notebook's own.
plt.rcParams.update(plt.rcParamsDefault)
LABEL_FS, TICK_FS = 16, 12

def style_axes(ax):
    ax.tick_params(which="both", direction="in", labelsize=TICK_FS, pad=6)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
%matplotlib inline

## Parameters

In [2]:
lg_psi_res = -2.5      # which saved tree: log10 tree mass resolution, Mres / M0
seed       = 12        # saved at this resolution: 8, 10, 12

time_axis  = 'log1pz'  # 'log1pz' (log10(1+z), time runs left to right), 'cosmic' (t [Gyr], redshift on top) or 'lookback'
xlim       = (0.7, 0.0)   # x limits in the units of time_axis (here log(1+z), early to late); None = fit the whole tree
ylim       = 5e10, 2e12         # (low, high) limits of M_host [Msun] in the MAH panel; None = fit the part of the history inside xlim

# --- figure size
figsize       = (8, 4)     # (width, height) of the whole figure [inches]
height_ratios = (1.5, 1)     # tree panel : MAH panel
hspace        = 0.05         # gap between the two panels, as a fraction of the mean panel height

# --- text
text_fontsize   = LABEL_FS   # axis labels, time arrow, z50 label (tick labels use TICK_FS)
number_fontsize = TICK_FS    # the merger numbers, in both panels
number_weight   = 'normal'   # 'bold' or 'normal'
number_box      = dict(boxstyle='round,pad=0.2', fc='w', ec='k', lw=1.0)   # white box, black outline, behind each number
number_color    = 'k'
number_row_gap  = 0.28       # tree panel: the two rows of numbers sit this far above / below the outermost node (tree units)
number_sep      = 6          # tree panel: smallest gap between two numbers in a row [pixels]
number_leader   = dict(arrowstyle='-', color='k', lw=0.8)   # thin line from each number to its node; None = no line
mah_number_offset = (-26, 20)   # MAH panel: each number sits this far (x, y) [pt] from the middle of its jump; negative x = to the left
tree_pad        = (0.55, 0.22)  # empty space (below, above) the rows of numbers, in tree units; the space below holds the legend

# --- legend
show_legend     = True
legend_on       = 'tree'     # 'fig' (anchored to the figure), 'tree' or 'mah' (inside that panel)
legend_loc      = 'lower left'      # which point of the legend box sits at legend_anchor (or which corner, if no anchor)
legend_anchor   = None              # (x, y) in figure fractions ('fig') or panel fractions ('tree', 'mah'); None = let loc place it
legend_ncol     = 1                 # 1 = one entry per line; 4 = everything on one line
legend_fontsize = 0.75 * LABEL_FS   # as the LSST legend
legend_fit      = True              # shrink the legend text if the box would be wider than the panel
legend_frame    = True              # draw a box around the legend
legend_framealpha = 1.0             # opacity of the box (1 = hides the guides behind it)
legend_labels   = {'host':   'host halo',                   # set an entry to None to leave it out
                   'k1':     '1st order',
                   'higher': '{order}',            # {order} -> '2nd order', '3rd order', ...
                   'mah':    None}                          # the black M_vir line

# --- arrow of time, along the bottom of the MAH panel
show_time_arrow = True
time_arrow_x    = (0.05, 0.3)   # start and end of the arrow, as fractions of the panel width
time_arrow_y    = 0.1           # height of the arrow, as a fraction of the panel height
time_arrow_text = 'time'         # word above the tail of the arrow

# --- z50 of the host: the snapshot where M_host is closest to half its final mass (as Tree_Reader.host_z50)
show_z50      = True
z50_on_tree   = False           # draw the line through the tree panel too (the label is always on the MAH panel)
z50_to_curve  = True            # MAH panel: stop the line where it meets the curve (False = full height)
z50_style     = dict(color='k', ls='--', lw=1.5)
z50_label     = r'$z_{{50}} = {z:.2f}$'     # {z} -> the value; doubled braces are literal
z50_label_y   = 0.30           # height of the label, as a fraction of the MAH panel
z50_label_side = 'right'       # which side of the line the label sits on
z50_mark      = dict(marker='o', ms=7, mfc='w', mec='k', mew=1.5, ls='')   # point on the curve where M_host = M_vir / 2; None = no point

# --- target M_vir of the tree: the host mass at z = 0, as a horizontal line in the MAH panel
show_Mvir     = False
Mvir_style    = dict(color='k', ls='-.', lw=1.2)
Mvir_label    = r'$M_{{\rm vir}} = {m}\,{{\rm M}}_\odot$'   # {m} -> the value, e.g. 10^{12}; doubled braces are literal
Mvir_label_x  = 0.02           # left edge of the label, as a fraction of the MAH panel width
Mvir_label_side = 'below'      # 'above' or 'below' the line

N_label    = 6         # number the N most massive k=1 mergers in both panels
min_ratio  = 0.0       # only draw k=1 branches with m_acc / M_host(t_acc) above this (0 = all)

ORDER_COLORS = {0: '#e41a1c',   # host
                1: '#377eb8',   # k = 1
                2: '#984ea3',   # k = 2
                3: '#ff7f00',   # k = 3
                4: '#a65628'}   # k = 4
ORDER_NAME   = {0: 'host', 1: '1st order', 2: '2nd order', 3: '3rd order', 4: '4th order'}
higher_alpha = 0.6     # opacity of the k >= 2 limbs and nodes
lw_range     = (0.8, 4.0)     # limb line width [pt], scaled with log m_acc
size_range   = (25.0, 450.0)  # leaf-node area [pt^2], scaled with log m_acc
host_node    = 70.0    # area of the host nodes on the trunk [pt^2]
limb_len     = (0.40, 1.0)    # height of a k=1 limb above/below the trunk, scaled with log m_acc (panel half-height = 1)
lean         = 0.09    # how far a full-length k=1 limb leans back in time, as a fraction of the time axis
sub_limbs    = [(0.060, 0.13), (0.010, 0.28), (0.075, 0.02)]   # (lean, height) of sub-branch limbs; siblings cycle through these
sub_len      = (0.6, 1.3)     # sub-branch limbs are scaled by this, with log m_acc
sub_shrink   = 0.6     # each further order is drawn this much smaller
guides_numbered_only = True   # dotted guides only for the numbered mergers (False = one for every k=1 merger)
guide_style  = dict(color='0.45', ls=':', lw=0.9)

workdir = f'{SATGEN}/notebooks/visualization/schematic'
treedir, figdir = f'{workdir}/trees/', f'{workdir}/figures/'
tree_file = f'{treedir}tree_MW_lgres{lg_psi_res}_s{seed}_evo.npz'

## Load the tree

`t_start` is the first snapshot at which a branch has a mass (the leaf), and `acc` is the snapshot of its peak mass, when it merges with its parent (`Tree_Reader.acc_index`).

In [3]:
full = np.load(tree_file)
z, t = full['redshift'], full['CosmicTime']                 # index 0 is z = 0
mass   = np.delete(full['mass'], 1, axis=0)
order  = np.delete(full['order'], 1, axis=0)
parent = np.delete(full['ParentID'], 1, axis=0)
mass   = np.where(mass == -99, np.nan, mass)
parent[parent > 0] -= 1

Nhalo  = mass.shape[0]
idx    = np.arange(Nhalo)
acc    = np.nanargmax(mass, axis=1)                         # accretion snapshot
m_acc  = mass[idx, acc]
k_acc  = order[idx, acc]
p_acc  = parent[idx, acc]
start  = np.array([np.where(np.isfinite(mass[i]))[0].max() for i in idx])   # earliest snapshot

Mhost  = mass[0]
print(f'{tree_file.split("/")[-1]}: {Nhalo - 1} branches, ' +
      ', '.join(f'{np.sum(k_acc[1:] == k)} with k={k}' for k in np.unique(k_acc[1:])))

tree_MW_lgres-2.5_s12_evo.npz: 56 branches, 24 with k=1, 29 with k=2, 3 with k=3


## Match every $k=1$ branch to its jump

`dM[j]` is the host's growth arriving at snapshot `j`. The tree writes a merger into the host mass one or two snapshots after the branch's own accretion snapshot, so for each $k=1$ branch the jump is the `dM` in the few snapshots just after `acc` that best matches its $m_{\rm acc}$ (most massive branches are matched first, and what they use is subtracted). Branches that land in the same snapshot share one jump.

The table is the check that this works: for every jump, the summed $m_{\rm acc}$ of its branches should equal `dM` up to the smooth accretion in that step (ratio $\approx 1$).

In [4]:
dM = np.zeros_like(Mhost)
dM[:-1] = Mhost[:-1] - Mhost[1:]                # growth between snapshot j+1 and j
dM = np.nan_to_num(dM)

k1 = np.where(k_acc == 1)[0]
k1 = k1[m_acc[k1] / Mhost[acc[k1]] > min_ratio]
jump = {}                                       # branch -> snapshot of its jump
left = dM.copy()                                # growth not yet assigned to a branch
for i in k1[np.argsort(m_acc[k1])[::-1]]:       # most massive first
    cand = np.arange(max(acc[i] - 3, 0), acc[i] + 1)
    excess = np.log10(np.clip(left[cand], 1.0, None) / m_acc[i])
    cost = np.where(excess > -0.02, excess, 10 - excess)   # a jump is m_acc plus smooth accretion, so never smaller than m_acc
    j = cand[np.argmin(cost)]
    jump[i] = int(j)
    left[j] -= m_acc[i]

events = {}                                     # jump snapshot -> its branches
for i, j in jump.items():
    events.setdefault(j, []).append(i)

# number the most massive k=1 mergers
ranked = k1[np.argsort(m_acc[k1])[::-1]][:N_label]
number = {i: n + 1 for n, i in enumerate(ranked)}

print('   z      t   branches  log sum(m_acc)  log dM   ratio')
for j in sorted(events, reverse=True):
    s = m_acc[events[j]].sum()
    print(f'{z[j]:5.2f} {t[j]:6.2f}   {len(events[j]):3d}       {np.log10(s):6.2f}      {np.log10(dM[j]):6.2f}   {s / dM[j]:5.2f}')

discrete = sum(m_acc[i] for i in k1)
print(f'\nk=1 mergers: {discrete / Mhost[0]:.0%} of the final mass; the rest is smooth (sub-resolution) accretion')

   z      t   branches  log sum(m_acc)  log dM   ratio
 6.79   0.78     1         9.94        9.97    0.93
 5.56   1.01     1         9.56        9.69    0.75
 5.28   1.08     1         9.66        9.76    0.79
 5.08   1.13     1         9.73        9.77    0.90
 4.96   1.17     1         9.58        9.70    0.77
 3.60   1.71     1        10.44       10.46    0.96
 2.59   2.48     1        10.62       10.64    0.96
 2.15   3.00     1         9.83        9.94    0.78
 2.05   3.15     1        10.83       10.84    0.97
 1.71   3.75     1         9.57        9.76    0.65
 1.49   4.23     1         9.65        9.81    0.69
 1.27   4.83     1        10.52       10.54    0.94
 1.25   4.89     1        10.97       10.97    0.99
 1.23   4.95     1        10.34       10.38    0.91
 0.95   5.97     1        10.63       10.65    0.95
 0.89   6.21     1         9.93       10.00    0.86
 0.70   7.17     1         9.51        9.67    0.70
 0.69   7.23     1         9.88        9.96    0.84
 0.51   8

## Layout of the tree

Time is the only physical coordinate; y just keeps the limbs apart.

* The trunk is the line $y=0$, and **where a $k=1$ limb meets it is physical**: the time of that merger, which is the time of its jump in the lower panel.
* The rest is schematic, as in `plot_acc_tree`, where x only keeps branches apart. Every limb leans back in time from its merge point to its leaf node. $k=1$ limbs alternate above and below the trunk and are all parallel, so they cannot cross; more massive mergers get longer limbs.
* Sub-branches attach along their parent limb in order of merger time (earliest nearest the tip), and leave it at one of a few angles (`sub_limbs`).

In [5]:
def xof(j):
    # x coordinate of snapshot(s) j on the chosen time axis
    if time_axis == 'cosmic':   return t[j]
    if time_axis == 'lookback': return t[0] - t[j]
    if time_axis == 'log1pz':   return np.log10(1 + z[j])
    raise ValueError(time_axis)

valid = np.where(np.isfinite(Mhost))[0]
span  = np.ptp(xof(valid))

end = acc.copy()                               # snapshot at which each branch merges
for i, j in jump.items():
    end[i] = j

def layout(show_higher_order):
    # positions of every limb; returns a namespace the figure function reads
    drawn = set(k1)
    if show_higher_order:
        for k in range(2, order.max() + 1):
            drawn |= {i for i in np.where(k_acc == k)[0] if p_acc[i] in drawn}
    kids = {i: sorted((c for c in drawn if p_acc[c] == i), key=lambda c: -end[c]) for i in drawn | {0}}   # earliest merger first

    lm = np.log10(m_acc)
    def scaled(i, lo_hi):
        f = (lm[i] - lm[sorted(drawn)].min()) / max(np.ptp(lm[sorted(drawn)]), 1e-9)
        return lo_hi[0] + f * (lo_hi[1] - lo_hi[0])

    back = -np.sign(xof(0) - xof(1)) * span        # one time-axis length, pointing back in time
    leaf, merge, side = {}, {}, {}                 # branch -> (x, y) of its two ends, and its side of the trunk

    def grow(i, shrink):
        # place the sub-branches of limb i along it, then recurse
        (x0, y0), (x1, y1) = leaf[i], merge[i]
        for n, c in enumerate(kids[i]):
            f = 0.25 + 0.68 * (n + 0.5) / len(kids[i])         # earliest merger nearest the tip (f = 0)
            dx, dy = np.array(sub_limbs[n % len(sub_limbs)]) * scaled(c, sub_len)
            side[c]  = side[i]
            merge[c] = (x0 + f * (x1 - x0), y0 + f * (y1 - y0))
            leaf[c]  = (merge[c][0] + dx * shrink * back, merge[c][1] + side[i] * dy * shrink)
            grow(c, shrink * sub_shrink)

    for n, i in enumerate(kids[0]):                # k=1: alternate sides, in order of merger time
        side[i]  = 1 if n % 2 == 0 else -1
        h        = scaled(i, limb_len)
        merge[i] = (xof(end[i]), 0.0)
        leaf[i]  = (merge[i][0] + lean * h * back, side[i] * h)
        grow(i, 1.0)
    return SimpleNamespace(drawn=drawn, kids=kids, leaf=leaf, merge=merge, side=side, scaled=scaled)

## Figure

In [6]:
def box_px(txt):
    # pixel extent of an annotation's text box, without its leader line
    patch = txt.get_bbox_patch()
    return patch.get_window_extent() if patch is not None else plt.Text.get_window_extent(txt)

def shift_px(ax, txt, dx, dy):
    # move an annotation's text by (dx, dy) pixels, whatever units its position is in
    if dx == 0 and dy == 0:
        return
    if txt.anncoords == 'data':
        px = ax.transData.transform(txt.xyann)
        txt.xyann = tuple(ax.transData.inverted().transform(px + [dx, dy]))
    else:                                   # 'offset points'
        to_pt = 72.0 / ax.figure.dpi
        txt.xyann = (txt.xyann[0] + dx * to_pt, txt.xyann[1] + dy * to_pt)

def keep_inside(fig, ax, texts, pad=4):
    # nudge each annotation so that its box stays inside the frame of `ax` (pad in pixels)
    fig.canvas.draw()
    frame = ax.get_window_extent()
    for txt in texts:
        bb = box_px(txt)
        shift_px(ax, txt, max(frame.x0 + pad - bb.x0, 0) + min(frame.x1 - pad - bb.x1, 0),
                          max(frame.y0 + pad - bb.y0, 0) + min(frame.y1 - pad - bb.y1, 0))

def spread_row(fig, ax, texts, sep=6, pad=4):
    # slide the numbers of one row sideways until none overlap (sep, pad in pixels); they stay on their row
    if not texts:
        return
    fig.canvas.draw()
    frame = ax.get_window_extent()
    boxes = sorted(([box_px(txt).x0, box_px(txt).x1, txt] for txt in texts), key=lambda b: b[0] + b[1])
    orig  = [b[0] for b in boxes]
    for _ in range(200):
        moved = False
        for a, b in zip(boxes[:-1], boxes[1:]):
            overlap = a[1] + sep - b[0]
            if overlap > 0.01:                              # push the pair apart, half each
                a[0] -= overlap / 2; a[1] -= overlap / 2
                b[0] += overlap / 2; b[1] += overlap / 2
                moved = True
        for b, edge in ((boxes[0], frame.x0 + pad - boxes[0][0]), (boxes[-1], frame.x1 - pad - boxes[-1][1])):
            if (b is boxes[0] and edge > 0.01) or (b is boxes[-1] and edge < -0.01):   # keep the ends inside the frame
                b[0] += edge; b[1] += edge
                moved = True
        if not moved:
            break
    for b, x0 in zip(boxes, orig):
        shift_px(ax, b[2], b[0] - x0, 0)

def declutter(fig, ax, texts, sep=4, pad=4):
    # MAH numbers, in the order given: if a number overlaps one already placed, lift it clear;
    # if that would leave the panel, slide it further left instead (sep, pad in pixels)
    fig.canvas.draw()
    frame, placed = ax.get_window_extent(), []
    for txt in texts:
        for _ in range(20):
            fig.canvas.draw()
            bb = box_px(txt)
            hit = [o for o in placed if bb.x0 < o.x1 + sep and o.x0 < bb.x1 + sep and bb.y0 < o.y1 + sep and o.y0 < bb.y1 + sep]
            if not hit:
                break
            up = max(o.y1 for o in hit) + sep - bb.y0
            if bb.y1 + up <= frame.y1 - pad:
                shift_px(ax, txt, 0, up)
            else:
                shift_px(ax, txt, min(o.x0 for o in hit) - sep - bb.x1, 0)
        keep_inside(fig, ax, [txt], pad)
        fig.canvas.draw()
        placed.append(box_px(txt))

def draw(show_higher_order, tag):
    # the two-panel figure; saved as merger_tree_MAH_<tag>_lgres..._s....pdf
    S = layout(show_higher_order)
    drawn, leaf, merge, side, scaled = S.drawn, S.leaf, S.merge, S.side, S.scaled
    num_kw = dict(textcoords='offset points', fontsize=number_fontsize, fontweight=number_weight, color=number_color,
                  bbox=number_box, zorder=30)
    tree_kw = dict(num_kw, textcoords='data', ha='center', va='center', arrowprops=number_leader)
    ys = np.array([leaf[i][1] for i in drawn])
    row_y = {1: max(ys.max(), 0.3) + number_row_gap, -1: min(ys.min(), -0.3) - number_row_gap}   # the two rows of numbers
    tree_numbers, mah_numbers = {1: [], -1: []}, []

    fig, (axT, axM) = plt.subplots(2, 1, figsize=figsize, sharex=True,
                                   gridspec_kw=dict(height_ratios=height_ratios, hspace=hspace))

    # ---------------------------------------------------------------- merger tree
    axT.plot(xof(valid), np.zeros(valid.size), color=ORDER_COLORS[0], lw=5, solid_capstyle='butt', zorder=3)   # trunk

    for i in sorted(drawn, key=lambda i: -k_acc[i]):          # k=1 drawn last, on top
        k = int(k_acc[i])
        a = 1.0 if k == 1 else higher_alpha
        (x0, y0), (x1, y1) = leaf[i], merge[i]
        axT.plot([x0, x1], [y0, y1], color=ORDER_COLORS[k], lw=scaled(i, lw_range), alpha=a,
                 solid_capstyle='round', zorder=4 if k == 1 else 2)
        axT.scatter(x0, y0, s=scaled(i, size_range), color=ORDER_COLORS[k], alpha=a, ec='k', lw=0.6,
                    zorder=7 if k == 1 else 5)
        if i in number:
            # on the row of its side of the trunk, straight above / below its leaf node
            tree_numbers[side[i]].append(axT.annotate(str(number[i]), (x0, y0), xytext=(x0, row_y[side[i]]), **tree_kw))

    # host nodes: where each k=1 limb joins the trunk
    for j in sorted(events):
        axT.scatter(xof(j), 0, s=host_node, color=ORDER_COLORS[0], ec='k', lw=0.6, zorder=6)

    # ---------------------------------------------------------------- host MAH
    axM.plot(xof(valid), Mhost[valid], color='k', lw=1.5, drawstyle='steps-pre', zorder=3)
    # (snapshots run from z=0 backwards; 'steps-pre' puts each step at snapshot j, where the mass arrives)

    for j in sorted(events, reverse=True):                    # early to late
        x, lo, hi = xof(j), Mhost[j + 1], Mhost[j]
        axM.plot([x, x], [lo, hi], color=ORDER_COLORS[1], lw=3.5, solid_capstyle='butt', zorder=4)   # the jump
        labels = sorted(number[i] for i in events[j] if i in number)
        if labels or not guides_numbered_only:
            # guide: from the trunk, through the bottom of the tree panel, down to the top of the jump
            axT.plot([x, x], [0, -10], zorder=1, **guide_style)   # from the host node down to the edge of the panel
            axM.plot([x, x], [hi, 1e30], zorder=1, **guide_style)
        if labels:
            # up and to the left of the jump (the empty side of the curve), with a thin line back to its middle
            mah_numbers.append(axM.annotate(
                ', '.join(map(str, labels)), (x, np.sqrt(lo * hi)), xytext=mah_number_offset,
                ha='center', va='center', arrowprops=number_leader, **num_kw))

    # ---------------------------------------------------------------- axes
    axT.set_ylim(row_y[-1] - tree_pad[0], row_y[1] + tree_pad[1])
    axT.set_yticks([])
    axT.set_ylabel('Merger Tree', fontsize=text_fontsize)

    axM.set_yscale('log')
    axM.set_ylabel(r"$M_{\rm vir}$", fontsize=text_fontsize)

    if time_axis == 'cosmic':
        axM.set_xlabel('cosmic time [Gyr]', fontsize=text_fontsize)
        axM.set_xlim(min(0, min(p[0] for p in leaf.values()) - 0.2), t[0] * 1.01)
        zticks = np.array([0, 0.25, 0.5, 1, 2, 3, 5, 10])
        top = axT.twiny(); top.set_xlim(axM.get_xlim())   # redshift ticks on a twin of the tree panel
        top.set_xticks(np.interp(zticks, z, t)); top.set_xticklabels([f'{v:g}' for v in zticks])
        top.set_xlabel('redshift', fontsize=text_fontsize); top.tick_params(which='both', bottom=False, labelsize=text_fontsize)
        top.tick_params(axis='y', which='both', left=False, right=False)
        axM.set_xticks(np.arange(0, 14, 2))
    elif time_axis == 'lookback':
        axM.set_xlabel('lookback time [Gyr]', fontsize=text_fontsize)
        axM.set_xlim(t[0] * 1.01, -0.1)
    else:
        axM.set_xlabel(r'$\log(1+z)$', fontsize=text_fontsize)
        axM.set_xlim(max(xof(valid.max()), max(p[0] for p in leaf.values())) * 1.03, -0.04)
    axT.tick_params(axis='x', which='both', top=False, labelbottom=False)
    for ax in (axT, axM):
        style_axes(ax)

    # limits: xlim / ylim from the parameters, otherwise fitted
    if xlim is not None:
        axM.set_xlim(*xlim)
    if ylim is not None:
        axM.set_ylim(*ylim)
    else:
        inside = valid[(xof(valid) >= min(axM.get_xlim())) & (xof(valid) <= max(axM.get_xlim()))]
        axM.set_ylim(0.6 * Mhost[inside].min(), 2.5 * Mhost[0])
    if time_axis == 'cosmic':
        top.set_xlim(axM.get_xlim())

    # ---------------------------------------------------------------- legend
    if show_legend:
        dot = lambda k, a=1.0: dict(marker='o', ls='', ms=0.75 * legend_fontsize, mfc=ORDER_COLORS[k], mec='k', mew=0.6, alpha=a)   # symbol only
        entries = [('host', Line2D([], [], **dot(0))), ('k1', Line2D([], [], **dot(1)))]
        entries += [('higher', Line2D([], [], label=ORDER_NAME[k], **dot(k, higher_alpha)))
                    for k in sorted({int(k_acc[i]) for i in drawn} - {1})]
        entries.append(('mah', Line2D([], [], color='k', lw=1.5)))
        handles = []
        for key, h in entries:
            if legend_labels.get(key) is None:
                continue
            h.set_label(legend_labels[key].format(order=h.get_label()) if key == 'higher' else legend_labels[key])
            handles.append(h)
        target = {'fig': fig, 'tree': axT, 'mah': axM}[legend_on]
        kw = dict(handles=handles, loc=legend_loc, ncol=legend_ncol, frameon=legend_frame, fontsize=legend_fontsize,
                  framealpha=legend_framealpha, facecolor='w', edgecolor='k', fancybox=False,
                  handletextpad=0.1, columnspacing=0.9, borderaxespad=0.4)
        if legend_anchor is not None:
            kw['bbox_to_anchor'] = legend_anchor
        leg = target.legend(**kw)
        if legend_fit and legend_on != 'fig':               # shrink the text until the box fits inside the panel
            fig.canvas.draw()
            while leg.get_window_extent().width > 0.98 * target.get_window_extent().width and kw['fontsize'] > 6:
                kw['fontsize'] -= 0.5
                for h in handles:
                    h.set_markersize(0.75 * kw['fontsize'])
                leg.remove()
                leg = target.legend(**kw)
                fig.canvas.draw()
        leg.set_zorder(20)

    # ---------------------------------------------------------------- z50 of the host
    if show_z50:
        j50 = int(np.nanargmin(np.abs(Mhost - 0.5 * Mhost[0])))     # same definition as Tree_Reader.host_z50
        if z50_on_tree:
            axT.axvline(xof(j50), zorder=2, **z50_style)
        if z50_to_curve:                                             # from the bottom of the panel up to the curve
            axM.plot([xof(j50)] * 2, [axM.get_ylim()[0], Mhost[j50]], zorder=2, **z50_style)
        else:
            axM.axvline(xof(j50), zorder=2, **z50_style)
        right = z50_label_side == 'right'
        axM.annotate(z50_label.format(z=z[j50]), (xof(j50), z50_label_y), xycoords=('data', 'axes fraction'),
                     xytext=(7 if right else -7, 0), textcoords='offset points', ha='left' if right else 'right',
                     va='center', fontsize=text_fontsize, zorder=25)
        if z50_mark is not None:                                     # where the curve crosses half the final mass
            axM.plot(xof(j50), Mhost[j50], zorder=26, **z50_mark)

    # ---------------------------------------------------------------- target M_vir (host mass at z = 0)
    if show_Mvir:
        M0 = Mhost[0]
        e = int(np.floor(np.log10(M0))); mant = M0 / 10**e
        m_txt = f'10^{{{e}}}' if abs(mant - 1) < 0.05 else f'{mant:.1f}\\times10^{{{e}}}'
        axM.axhline(M0, zorder=2, **Mvir_style)
        above = Mvir_label_side == 'above'
        axM.annotate(Mvir_label.format(m=m_txt), (Mvir_label_x, M0), xycoords=('axes fraction', 'data'),
                     xytext=(0, 4 if above else -4), textcoords='offset points', ha='left',
                     va='bottom' if above else 'top', fontsize=text_fontsize, zorder=25)

    # ---------------------------------------------------------------- arrow of time
    if show_time_arrow:
        (xa, xb), ya = time_arrow_x, time_arrow_y
        axM.annotate('', xy=(xb, ya), xytext=(xa, ya), xycoords='axes fraction', textcoords='axes fraction',
                     arrowprops=dict(arrowstyle='-|>', color='k', lw=1.5, mutation_scale=20), zorder=25)
        axM.annotate(time_arrow_text, (xa, ya), xycoords='axes fraction', textcoords='offset points', xytext=(0, 5),
                     ha='left', va='bottom', fontsize=text_fontsize, zorder=25)

    # numbers: spread each tree row so none overlap; keep the MAH numbers inside the panel and off each other
    for row in tree_numbers.values():
        spread_row(fig, axT, row, sep=number_sep)
    declutter(fig, axM, mah_numbers)

    fig.tight_layout()
    fig.subplots_adjust(hspace=hspace)              # keep the requested gap between the panels
    stem = f'{figdir}merger_tree_MAH_{tag}_lgres{lg_psi_res}_s{seed}'
    fig.savefig(f'{stem}.pdf', bbox_inches='tight')
    plt.show()
    return fig

## Version 1: first-order branches only

Host trunk plus the $k=1$ branches: one limb per jump.

In [ ]:
fig = draw(show_higher_order=False, tag='k1')

## Version 2: with the higher-order branches

The same tree and the same $k=1$ limbs, with the $k\geq2$ sub-branches added.

In [ ]:
fig = draw(show_higher_order=True, tag='all')